# 08 · SkewerJAX and fake_spectra: differences, accuracy and speed

fake_spectra is the external reference code for spectra from simulations. This notebook compares the two codes on
the same TNG50-4 skewers, in two parts:

- **Part 1, the differences step by step.** `skewerjax.comparison` re-implements fake_spectra's scheme on SkewerJAX's
  absorbers and runs fake_spectra itself.
  * **Matched mode:** `FAKE_SPECTRA_EXTRACTION` + `FAKE_SPECTRA_INTEGRATION` reproduce fake_spectra (top-hat spheres,
    its mass and b constants, its star-forming rule, 7-point chords, per-cell pixel average, linear detuning,
    `tautail`, one redshift with a linear Hubble flow).
  * **Default:** SkewerJAX's choices. Every difference is one named switch, and the **effect chain** goes from one to
    the other one switch at a time.
  * `FakeSpectraSource` gives fake_spectra's own spectra and per-cell inputs, **live** (fake_spectra installed; each run
    in its own process with `python -m skewerjax.comparison.driver full …`, cached) or from a **reference** HDF5 file.
- **Part 2, speed and memory.** The benchmark results saved by `scripts/benchmarks/benchmark_vs_fake_spectra.py`
  (read from disk, not re-run).

The full set of checks is `scripts/validation/check_fake_spectra.py`. Runtime: ~1 min (live).

**Requires:** simulation outputs (04), τ (06) and the external reference code fake_spectra (live, or its reference file).  
**Independent of:** the statistics (07).

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import (use_style, ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, LABELS, FIGSIZES,
                                      add_panel_letters, plot_dag, plot_spectra, zoom_inset)
use_style()                                        # the dark, colour-blind-safe style of every figure (skewerjax.utils.plotting)
%matplotlib inline
import tempfile
from skewerjax.core import LineBundle
from skewerjax.simulations import ArepoHDF5Reader, SimulationSkewers, ExtractionOptions, HI
from skewerjax.comparison import (FakeSpectraSource, SKEWER_SETS, FAKE_SPECTRA_EXTRACTION as EF, FAKE_SPECTRA_INTEGRATION as IF,
                                  DEFAULT_INTEGRATION, INTEGRATION_EFFECTS, EXTRACTION_EFFECTS, matched_tau, run_effects, effect_rows,
                                  spectral_metrics)
CACHE = tempfile.mkdtemp(prefix="fs_cache_")             # where live fake_spectra results go (or pass reference="file.h5")
DATA = os.environ.get("SKEWERJAX_DATA", "/home/STORAGE")      # folder with TNG50-4/ (docs/installation.md)
FS = FakeSpectraSource(cache=CACHE, snap_base=f"{DATA}/TNG50-4", snapnum=25)
print("fake_spectra:", FS.mode)
reader = ArepoHDF5Reader(f"{DATA}/TNG50-4", 25)
lya = LineBundle(["HI-LyA"], ref_line_id="HI-LyA")
sk = SimulationSkewers.axis_aligned(reader, SKEWER_SETS["single"]["cofm"], SKEWER_SETS["single"]["axis"], 0.25)
v, dv = sk.spectral_grid(1.0)                            # fake_spectra's periodic pixel grid
fs = FS.run("single_matched", "single", products="tau,colden")
t_matched = matched_tau(sk, HI, lya, v, EF, IF)
t_default = sk.tau(HI, lya, v)                            # SkewerJAX default (exact frame, on its own period)
print({k: f"{x:.1e}" for k, x in spectral_metrics(t_matched, fs["tau"], dv).items()})

In [ ]:
plot_dag(('simulation', 'tau', 'fake_spectra'), external=True); plt.show()

## Part 1 · The differences, step by step

Every difference between the two codes is one named switch. They act at two levels: how the gas is **extracted**
from the cells (layers 3 and 5 of notebook 04), and how the optical-depth integral is **evaluated** (layer 6). The
list below is printed from the package (`comparison.EXTRACTION_EFFECTS`, `INTEGRATION_EFFECTS`), with where each
choice comes from:

| switch | fake_spectra | SkewerJAX default | where it comes from |
|---|---|---|---|
| T constants | its m_p, k_B (`unitsystem.py`) | CODATA 2018 | `gas_physics.temperature`; T 8.1e-6 lower |
| b constants | its constants in b | CODATA, m_u | `comparison.effects`; b² ratio `FAKE_SPECTRA_B2_RATIO` |
| mass constant | n = ρX/(A m_p) | n = ρX/(A m_u) | `absorption.h` L4 labels PROTONMASS "1 a.m.u."; 0.73 % |
| neutral fraction (H I) | `sf_neutral` rule: ρ/m_p threshold | stored x_HI, star-forming cells recomputed | `gas_properties.get_reproc_HI` L116–L146 |
| CLOUDY density (metals) | hden = 0.774 ρ/m_p | n_H | `convert_cloudy.py` L183 |
| CLOUDY interpolation | cubic spline | bilinear in log f | `CloudyTable.ion` |
| deposition | top-hat spheres of radius V^{1/3} | Voronoi cells | `absorption.cpp:tophat_kern_frac` L112–L118 |
| tautail | stop at τ < 1e-7 per cell | full profiles | `absorption.cpp` L264, `spectra.py` L145 |
| pixel average | average over the pixel if b < 2 dv | pixel centres | `singleabs.h:tau_kern_outer` L133–L155 |
| detuning | linear, H(a, Δu/b) | exact, ν = (c/λ) expm1(−Δv/c) | `singleabs.h` L182; notes `app_velocity_units_approx.tex` |
| 7/8 normalisation | 7 interior chord points of weight 1/8 | (the exact chord) | `singleabs.h` L100–L118: a top hat integrates 7/8 of the column |
| chord quadrature | 7 points per chord | exact pieces of the dual segments | `comparison.emulation.fake_chord_absorbers` |
| frame | one a, u = aHs + v_pec | a(x) along the path, v_† | `spectra.py` L228 (v_max = aHL) |

**How each is quantified.** The *chain* switches fake_spectra's options to SkewerJAX's one at a time (integration on
fake_spectra's extraction, then the extraction on SkewerJAX's integration): it telescopes exactly, so the steps add up
to the total difference. *Alone* applies each switch by itself to fake_spectra's options, which isolates it.

In [ ]:
from skewerjax.comparison import EXTRACTION_EFFECTS as _EE, INTEGRATION_EFFECTS as _IE
for e in _EE + _IE:
    print(f"{e.kind:11s} {e.label:22s} ions={e.ions:6s} {e.what}")

In [ ]:
F_fs, F_m, F_d = np.exp(-fs["tau"][0]), np.exp(-t_matched[0]), np.exp(-t_default[0])
fig, axs = plt.subplots(2, 1, figsize=(16, 8), sharex=True, gridspec_kw={"height_ratios": [2, 1]}, layout="constrained")
axs[0].plot(v, F_fs, color=EC["fake_spectra"], **MS["reference"], label="fake_spectra")
axs[0].plot(v, F_m, color=EC["HI"], **MS["matched"], label="SkewerJAX, matched")
axs[0].plot(v, F_d, color=EC["HI"], **MS["default"], alpha=0.5, label="SkewerJAX, default")
axs[1].plot(v, F_m - F_fs, color=EC["HI"], **MS["matched"], label="matched − fake_spectra")
axs[1].plot(v, F_d - F_fs, color=EC["HI"], **MS["default"], alpha=0.5, label="default − fake_spectra")
axs[0].set(ylabel="F", xlim=(1500, 2500)); axs[1].set(xlabel="u [km/s]", ylabel=r"$\Delta F$", ylim=(-1, 1))
axs[1].set_yscale("symlog", linthresh=1e-5); axs[1].set_yticks([-1e-1, -1e-3, -1e-5, 0, 1e-5, 1e-3, 1e-1])
for ax in axs: ax.legend(fontsize=13)
add_panel_letters(axs); fig.suptitle("One TNG50-4 skewer: matched mode reproduces fake_spectra (|ΔF| ~ 10⁻⁵); the default differs", fontsize=21); plt.show()

### Step 1: the gas, before any integration

The column per position pixel isolates the extraction from the integration. fake_spectra's own `colden`, SkewerJAX's
top-hat deposition with fake_spectra's constants (matched extraction) and SkewerJAX's default Voronoi deposition, on
the same skewer: the matched columns agree to the float32 floor, and the Voronoi cells move the gas between pixels
without changing the total much.

In [ ]:
N_fs = np.asarray(fs["colden"][0])
N_th = sk.column_per_pixel(HI, EF, nbins=v.size)[0]
N_vo = sk.column_per_pixel(HI, ExtractionOptions(), nbins=v.size)[0]
fig, axs = plt.subplots(2, 1, figsize=(16, 8), sharex=True, gridspec_kw={"height_ratios": [2, 1]}, layout="constrained")
axs[0].semilogy(v, N_fs, color=EC["fake_spectra"], **MS["reference"], label="fake_spectra colden")
axs[0].semilogy(v, N_th, color=EC["tophat"], **MS["matched"], label="SkewerJAX, top-hat (matched extraction)")
axs[0].semilogy(v, N_vo, color=EC["voronoi"], **MS["default"], alpha=0.6, label="SkewerJAX, Voronoi (default)")
has = N_fs > 0
axs[1].plot(v[has], N_th[has] / N_fs[has] - 1, color=EC["tophat"], **MS["matched"], label="top-hat / fake_spectra − 1")
axs[1].plot(v[has], N_vo[has] / N_fs[has] - 1, color=EC["voronoi"], **MS["default"], alpha=0.6, label="Voronoi / fake_spectra − 1")
axs[1].set_yscale("symlog", linthresh=1e-6); axs[1].set_yticks([-1, -1e-3, -1e-6, 0, 1e-6, 1e-3, 1]); axs[0].set(ylabel=r"$N_{\rm HI}$ per pixel [cm$^{-2}$]", xlim=(1500, 2500))
axs[1].set(xlabel="pixel position [km/s of Hubble flow]", ylabel="relative difference")
for ax in axs: ax.legend(fontsize=12)
add_panel_letters(axs); fig.suptitle("The extraction alone: H I column per pixel on the skewer of the figure above", fontsize=21); plt.show()
print(f"total N_HI: fake_spectra {N_fs.sum():.4e} | top-hat {N_th.sum():.4e} ({N_th.sum() / N_fs.sum() - 1:+.1e}) | "
      f"Voronoi {N_vo.sum():.4e} ({N_vo.sum() / N_fs.sum() - 1:+.1e}); median per-pixel |top-hat / fs - 1| = {np.median(np.abs(N_th[has] / N_fs[has] - 1)):.1e}")

### Which differences matter? The effect chain

The chain switches fake_spectra's options to SkewerJAX's one at a time (integration on fake_spectra's fields, then
the extraction); "alone" applies each switch by itself. The chain telescopes exactly, so the bars add up to the
total difference.

In [ ]:
res_i = run_effects(sk, HI, lya, v, INTEGRATION_EFFECTS, EF, IF, verbose=False)
res_e = run_effects(sk, HI, lya, v, EXTRACTION_EFFECTS, EF, DEFAULT_INTEGRATION, verbose=False)
rows = effect_rows(res_i, dv)[0] + effect_rows(res_e, dv)[0]
fig, ax = plt.subplots(figsize=(15, 7), layout="constrained")
y = np.arange(len(rows))
ax.barh(y + 0.2, [r["dF_chain"] for r in rows], 0.4, color=TOL["blue"], label="along the chain")
ax.barh(y - 0.2, [r["dF_iso"] for r in rows], 0.4, color=TOL["yellow"], label="alone")
ax.set_yticks(y, [r["effect"] for r in rows]); ax.axvline(0, color=FG, lw=1)
ax.set_xscale("symlog", linthresh=1e-6); ax.set_xlabel(r"$\Delta\langle F\rangle$ of each switch (symmetric log scale)"); ax.legend()
fig.suptitle(f"Mean flux from fake_spectra to SkewerJAX: {sum(r['dF_chain'] for r in rows):+.4f} in total", fontsize=21); plt.show()

The same chain as a table: Δ⟨F⟩ and the max |ΔF| of each switch along the chain and alone, and the change of the P1D in two k bands (s/km).

In [ ]:
from skewerjax.comparison import print_effect_rows
rows_i, tot_i = effect_rows(res_i, dv)
rows_e, tot_e = effect_rows(res_e, dv)
print_effect_rows(rows_i, tot_i, "integration (on fake_spectra's extraction)")
print_effect_rows(rows_e, tot_e, "extraction (on SkewerJAX's integration)")

Each switch is an isolated option (`ExtractionOptions`, `IntegrationOptions`, `comparison.MatchedIntegration`), so any
combination can be evaluated with `matched_tau(sk, ion, lines, v, ext, integ)`. To use the comparison without
fake_spectra installed, export a reference file once in live mode (`FS.export("fake_spectra_reference.h5")`, or
`check_fake_spectra.py --export FILE`) and pass `FakeSpectraSource(reference=...)`.

### Which approximations matter, and what could be improved

The ranking is read off the table above (one skewer; `check_fake_spectra.py` and Part 2 for more).
- **The 7/8 normalisation (the largest shift of the mean flux).** The 7 interior chord points of weight 1/8 integrate 7/8
  of a top hat's column (`singleabs.h` L100–L118): every τ of fake_spectra is 12.5 % low before saturation. It is a
  quadrature artefact of an SPH-kernel scheme used with a top-hat kernel; weights 1/7, or the exact chord, remove it.
- **Deposition (the largest change of the spectra).** fake_spectra spreads each Voronoi cell over a sphere of radius
  V^{1/3}: neighbouring spheres overlap and leave gaps, so the gas along the sightline is multi-valued (several T and v
  at one position) and its column moves between pixels; it changes the P1D most at 0.1 < k < 1 s/km. SkewerJAX's
  default deposits Arepo's own cells exactly (04).
- **Frame.** One scale factor and a linear Hubble flow across the box hardly change ⟨F⟩ but move features (large
  max |ΔF|, a P1D change at high k); SkewerJAX integrates a(x) along the path (02, 03). The error grows with the box
  length in velocity.
- **Chord quadrature, mass and b constants.** At the 1e-3 level in ⟨F⟩: the 7-point chord against the exact pieces, and
  m_p against m_u (0.73 % in every density).
- **Detuning, pixel average, tautail.** The linear detuning is first order in Δv/c (negligible inside a line); the pixel
  average changes narrow metal lines when b < 2 dv; `tautail` truncates the far wings at τ < 1e-7 per cell. All three
  are small (table above).
- **Open.** The TNG50-2 Si III matched agreement (1.2e-3, Part 2) is above the 5e-5 of the other runs and has not been
  investigated (reply 20261021_2330).

## Part 2 · Speed and memory

The setup:
- **Inputs:** both codes compute Lyα and Si III λ1206 on identical skewers of the same TNG50 snapshots (snapshot 25,
  z = 3), at 1 km/s pixels. fake_spectra runs with its default (top-hat) kernel.
- **Process isolation:** every configuration ran in its own process, pinned to a fixed CPU set with a fixed thread
  count (`OMP_NUM_THREADS` for fake_spectra's OpenMP, the same CPU affinity for XLA).
- **Timing:** each process ran several repeats; the figures show medians with min–max bars. The wall time is split
  into setup, I/O, cell selection, gas physics and τ.
- **Agreement:** SkewerJAX's matched mode against fake_spectra's τ on the first skewers, and the mean flux of
  SkewerJAX's default mode.

This notebook only reads the saved results (`RESULTS_DIR`); the runs come from
`scripts/benchmarks/benchmark_vs_fake_spectra.py`. Runtime: seconds.

In [ ]:
import json
from skewerjax.comparison.benchmark import load_results, summary, STAGES
from skewerjax.utils.plotting import (plot_bench_scaling, plot_bench_stages, plot_bench_memory, plot_bench_speedup, plot_bench_accuracy,
                                      plot_bench_resolution, bench_select, bench_key, BENCH_STYLES)
RESULTS_DIR = os.environ.get("SKEWERJAX_BENCHMARK_RESULTS", "../claude_agents/tmp_files/20261021_2330/scratch/benchmarks")
R = load_results(RESULTS_DIR)
ok = [r for r in R if r.get("status") == "ok"]
print(f"{len(R)} results in {RESULTS_DIR}: {len(ok)} ok, {len(R) - len(ok)} timed out")
m = ok[0]["metadata"]
print(f"CPU: {m['cpu']} ({m['logical_cpus']} logical CPUs), {m['ram_GB']:.0f} GB RAM | GPU: {m['gpu']}")
print("software:", {k: v for k, v in m["versions"].items() if v}, "| Python", m["python"])

### Summary table

Median wall time [s] per stage (all repeats), JIT compilation of SkewerJAX's τ, peak host memory, and the matched-mode flux agreement (max |ΔF| on the first skewers, Lyα / Si III).

In [ ]:
rows = []
for r in sorted(ok, key=lambda r: (r["config"]["snap_base"], r["config"]["n_skewers"], r["config"]["threads"], str(bench_key(r)))):
    s, c, a = summary(r), r["config"], r.get("agreement") or {}
    rows.append([c["snap_base"].split("/")[-1], c["n_skewers"], c["threads"], BENCH_STYLES[bench_key(r)]["label"], s["n_repeats"],
                 *[f"{s[k][0]:.2f}" for k in STAGES], f"{s['total'][0]:.1f}", "" if r.get("compile_s") is None else f"{r['compile_s']:.1f}",
                 f"{r['peak_rss_GB']:.2f}", "/".join(f"{a[l]['max_dF']:.0e}" for l in ("HI-LyA", "SiIII-1206")) if a else ""])
head = ["sim", "N_sk", "thr", "code", "rep", *STAGES, "total", "compile", "RSS GB", "max|dF|"]
w = [max(len(str(x)) for x in col) for col in zip(head, *rows)]
for row in [head] + rows:
    print("  ".join(str(x).rjust(n) for x, n in zip(row, w)))
for r in R:
    if r.get("status") != "ok":
        print(f"not measured: {r['id']} ({r['status']} after {r['wall_s']:.0f} s)")

### Time against the number of skewers (TNG50-4, all 32 threads)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=FIGSIZES["wide"], layout="constrained")
plot_bench_scaling(axs[0], ok, sim="TNG50-4", threads=32, what="total"); axs[0].set_title("total")
plot_bench_scaling(axs[1], ok, sim="TNG50-4", threads=32, what="tau"); axs[1].set_title(r"$\tau$ only (first repeat includes the JIT compilation)")
axs[0].legend(fontsize=12); add_panel_letters(axs); plt.show()

*Takeaway:* from 1 to 256 skewers, fake_spectra goes from 1.6 s to 6.5 s and SkewerJAX (CPU) from 5.3 s to 447 s. SkewerJAX's τ costs about 1.6 s per skewer (two lines, 3961 pixels), fake_spectra's about 0.02 s, so the gap widens from 3.4× to 69×.

### Where the time goes (TNG50-4, 64 skewers)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=FIGSIZES["wide"], layout="constrained")
plot_bench_stages(axs[0], ok, sim="TNG50-4", n_skewers=64, threads=32)
plot_bench_stages(axs[1], ok, sim="TNG50-4", n_skewers=64, threads=32, fraction=True); axs[1].get_legend().remove()
add_panel_letters(axs); plt.show()

*Takeaway:* at 64 skewers, τ is 91 % of SkewerJAX's CPU time (103 of 114 s), and the cell selection (5.9 s against 0.08 s) is the second gap. SkewerJAX's I/O is the shorter one (0.31 s against 0.96 s).

### Memory

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=FIGSIZES["wide"], layout="constrained")
plot_bench_memory(axs[0], ok, sim="TNG50-4", threads=32); plot_bench_memory(axs[1], ok, sim="TNG50-4", threads=32, device=True)
axs[0].legend(fontsize=12); axs[1].set_title("GPU (XLA_PYTHON_CLIENT_PREALLOCATE=false)"); add_panel_letters(axs); plt.show()

*Takeaway:* fake_spectra peaks at 0.5 GB of host memory for every skewer count. SkewerJAX grows by about 60 MB per skewer (1.5 GB at 1 skewer, 5.1 GB at 64, 16 GB at 256), and the GPU peak grows from 0.14 GB to 3.5 GB.

### Speed-up, threads and resolution

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=FIGSIZES["triple"], layout="constrained")
plot_bench_speedup(axs[0], ok, sim="TNG50-4", threads=32, what="total")
plot_bench_scaling(axs[1], ok, x="threads", sim="TNG50-4", n_skewers=16, platform="cpu", dtype=None); axs[1].set_title("16 skewers, CPU")
plot_bench_resolution(axs[2], ok, n_skewers=16)
axs[0].legend(fontsize=11); axs[1].legend(fontsize=11); add_panel_letters(axs); plt.show()

*Takeaway:* no configuration beats fake_spectra.
- **Float32** cuts SkewerJAX's CPU τ by 2.5× (16 skewers).
- **Threads:** SkewerJAX's τ scales 7.8× from 1 to 8 performance cores, and then only 1.17× more up to 32 threads.
- **GPU:** the RTX 3060 is 1.8× slower than the CPU in float64, and 3.8× slower in float32.
- **Resolution:** the ratio falls to 1.5× at TNG50-2 resolution, where fake_spectra's reading of the 349 GB snapshot dominates and SkewerJAX's cell selection takes 248 s.

### Accuracy: did both codes compute the same thing?

Filled: SkewerJAX's matched mode against fake_spectra (max |ΔF|, first 4 skewers, float64). Open: the mean-flux difference of SkewerJAX's default mode (all skewers), i.e. the size of the modelling differences that Part 1 breaks down. Dotted: 5e-5.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=FIGSIZES["wide"], layout="constrained", sharey=True)
plot_bench_accuracy(axs[0], ok, line="HI-LyA"); plot_bench_accuracy(axs[1], ok, line="SiIII-1206")
axs[0].set_title(r"Ly$\alpha$"); axs[1].set_title("Si III 1206"); axs[1].set_ylabel("")
for ax in axs: ax.axhline(5e-5, color=FG, ls=":", lw=1)
axs[0].set_ylim(top=0.5); axs[0].legend(fontsize=12, loc="lower right")
add_panel_letters(axs); plt.show()

*Takeaway:* in matched mode, the two codes agree to max |ΔF| ≤ 2.3e-5 (Lyα) and 2.2e-7 (Si III) on TNG50-4. On TNG50-3 the figures are 2.8e-5 and 6.2e-5, and on TNG50-2 6.4e-5 and 1.2e-3. The TNG50-2 Si III value is above the dotted 5e-5 line, and its cause has not been investigated. SkewerJAX's default mode gives a lower Lyα mean flux (0.500 against 0.533 at 16 skewers), which Part 1 breaks down effect by effect.

## Summary

- **Agreement:** the matched mode reproduces fake_spectra to |ΔF| ~ 1e-5. The default differs by the sum of the named
  switches, each evaluated along the chain and alone; the deposition and the 7/8 chord normalisation dominate.
- **Speed:** fake_spectra is faster on this machine in every configuration measured, from 1.5× to 69×; the gap is
  SkewerJAX's τ kernel (an exact Voigt profile for every absorber at every pixel) and, at high resolution, its cell
  selection. float32 and more CPU cores help SkewerJAX; this GPU is slower than the CPU.
- Without fake_spectra installed, Part 1 runs from a reference HDF5 file; Part 2 only reads saved results
  (`SKEWERJAX_BENCHMARK_RESULTS`). Checks: `scripts/validation/check_fake_spectra.py`.